# HelioAI — the St Patrick's Day storm, 17 March 2015

The largest geomagnetic storm of solar cycle 24 (Dst minimum −223 nT), worked end to end:
find the parameters, download them, characterise the shock and the driver, and compute the
plasma regimes on either side.

This notebook is a **worked scientific example** rather than a feature tour — see
`01_jupyter_tour.ipynb` for that.

**Why this event, and not the 2003 Halloween storm**

Halloween 2003 is the more famous choice, and it was this notebook's first subject. It was
replaced because the archives cannot support it. Measured over 2003-10-28T20:00 →
2003-10-30T06:00: ACE/SWEPAM density, speed and temperature are **100 % fill values** — the
instrument saturated in the particle storm — Wind/SWE is 69–74 % fill, OMNI 1-min is 57 %
fill and its temperature 99.7 %, and SOHO/CELIAS, the one product that returns clean data,
clips its speed at a constant 1019 km/s, its instrument ceiling. No standard speasy product
reproduces that event's headline numbers, which is a large part of why it is famous. It
makes a poor first demonstration.

17 March 2015 has clean science-quality coverage from a single spacecraft, so every number
below is reproducible.

**All reference values here were measured from the archive, not quoted.** The script that
measures them is [`verify_reference_values.py`](verify_reference_values.py) — run it and
compare. Where a published value and the archive disagreed, the archive won. If your run
disagrees with a number in this notebook, the number in this notebook is wrong; please open
an issue.

**Before running this**

1. `pip install helioai` (or `uv sync` from a clone)
2. One LLM provider key in `.env` — see the
   [installation guide](https://erdoganfurkan.github.io/HelioAI/installation/)
3. `helioai index` once (~10 min)

> Outputs are stripped on purpose — run the cells to produce your own. Wall time is
> dominated by data download; expect a few minutes.


In [ ]:
%load_ext helioai.interfaces.jupyter_magic

import os

provider = os.environ.get("HELIOAI_LLM_PROVIDER", "azure")
print(f"LLM provider : {provider}")
print("Ready — run cells below with Shift+Enter.")

---
## Act I — Data Discovery

The prompts in this notebook are written the way you would actually brief a
colleague: the scientific goal and the constraints that matter, not a numbered
procedure. Where a prompt does say something specific, it is a fact about the
*data* the agent has no way to know — which archives are trustworthy for this
event, which structure is not the shock. Never a formula, and never a method.

That line is worth holding. A prompt that dictates the steps only demonstrates
that the model can follow steps, and Act III shows the concrete cost of crossing
it: the version of this notebook that supplied the Rankine-Hugoniot formulas
supplied a *wrong* one, and got a Mach number five times too large.

Finding the right parameter ids is the first real task. The same quantity appears
under dozens of names across ACE, Wind, DSCOVR and STEREO, with the usual traps:
`onboard` versus `prime` moments, `RTN` versus `GSE`, `HIA` versus `CODIF` for ions,
`MFI` versus `MAG` for magnetometers.

The one that bites hardest here is quality level. **`K0` products are browse
quality.** For this event `WI_K0_SWE/Np` carries no fill values at all and still
has single-point spikes to 166 cm⁻³ — enough to put a false shock four hours early
and return a compression ratio of 1.1. `WI_H1_SWE/*_moment` is the science product,
and it is clean. The prompt asks for science quality without naming the id.

Resolving these ids means querying the 83 000-entry RAG index, and the search is
iterative: a broad query returns plausible candidates, and narrowing it by quality
level and cadence is what separates the science product from the browse one. Watch
how many refinement passes it takes — that is the work this step is really doing.

In [ ]:
%%helioai
I am working on the St Patrick's Day storm of 17 March 2015, the big
one of cycle 24, with Dst around -223 nT. A CME-driven shock reached L1 early
on the 17th. My window is 2015-03-16T18:00 to 2015-03-18T12:00 UT.

Find me the solar wind for it: magnetic field vector in GSM, proton
density, bulk speed and temperature. I want the whole set from one
spacecraft so the jump conditions across the shock stay self-consistent,
and Wind is the natural choice. I also want the field from ACE, because
I will compare shock timing between the two later.

Three things I care about more than speed here. Give me science-quality
products rather than browse ones: key-parameter data is fine for a
quick look, not for shock analysis. Pick a cadence suited to shock
timing, a few seconds or so; I do not want the sub-second product,
it is a million points for this window and buys me nothing here. And tell me what the coverage
actually is in my window rather than assuming it is complete: I would
rather know now than discover a gap halfway through the analysis.

---
## Act II — Solar Wind Visualization

A fast forward shock crossed Wind at **04:01 UT on 17 March**, followed by a compressed
sheath, then the prolonged southward-Bz interval that drove the main storm phase around
midday.

The shock is easy to get wrong. A density jump alone is not a shock — there is a larger
density rise at 12:51 UT with no speed change at all, which is a compression structure
inside the driver. A fast forward shock requires **n, |B| and V to jump together**, and that
criterion picks 04:01 UT.

**Reference values, all measured from the archive over this window** — reproduce them with
`python examples/verify_reference_values.py`:

| Quantity | Measured | Product |
|:---|:---|:---|
| Shock crossing (n, &#124;B&#124;, V together) | 04:00:59 UT Mar 17 | `WI_H1_SWE` + `WI_H0_MFI` |
| Shock arrival at Wind, from &#124;B&#124; | 03:59:55 UT Mar 17 | `WI_H0_MFI/B3GSM` (3 s) |
| Shock arrival at ACE, from &#124;B&#124; | 04:04:25 UT Mar 17 | `AC_H0_MFI/BGSM` (16 s) |
| Upstream speed (1 h before) | 411.3 km/s | `WI_H1_SWE/Proton_V_moment` |
| Downstream speed (30 min after) | 514.1 km/s | idem |
| Peak speed in window | 657.4 km/s | idem |
| Upstream density | 17.43 cm⁻³ | `WI_H1_SWE/Proton_Np_moment` |
| Downstream density | 45.12 cm⁻³ | idem |
| Peak density in window | 57.9 cm⁻³ | idem |
| Upstream &#124;B&#124; | 10.00 nT | `WI_H0_MFI/BGSM` |
| Downstream &#124;B&#124; | 25.27 nT | idem |
| Max &#124;B&#124; on Mar 17 | 33.11 nT | idem |
| Min Bz GSM on Mar 17 | −26.05 nT at 12:19:30 UT | idem |

> Coverage in this window: Wind SWE H1 moments **0.0 %** invalid, ACE MAG 0.1 %, Wind MFI
> 2.4 % (counting a vector sample as lost if any component is). Nothing here needs
> gap-filling — which is the whole reason this event replaced Halloween 2003.

In [ ]:
%%helioai
Plot the storm from the Wind data you just found, over the full window.

Four stacked panels sharing a time axis, dark background: the field
magnitude with Bz overlaid in GSM, then density, speed and temperature.
Colour Bz by sign, since southward is what drives the storm and I want
to see it at a glance.

Mark the shock on every panel. Find it yourself from the data: it is a
fast forward shock, so density, field and speed all step up together
across it. Be careful, there is a bigger density rise later in the day
with no speed change at all. That one is a compression structure inside
the driver, not the shock, and it will fool a detector that only watches
density. Then shade the sheath and mark where the sustained southward Bz
begins, since that is the storm's main phase onset.

Underneath, give me the numbers I will need next: the shock time, the
upstream and downstream averages of all four quantities over sensible
windows either side, and the minimum Bz with its time. Tell me what
averaging windows you chose and why.

---
## Act III — Rankine-Hugoniot Shock Physics

The prompt below asks for the physics, not for a procedure. It hands over no
formulas — HelioAI ships a `rankine_hugoniot` recipe, and `list_recipes` /
`load_recipe` are tools the agent can reach for on its own. If it has to be told
the jump conditions, it is not doing the work; it is substituting into an equation
someone else supplied.

That is not a stylistic preference. **The earlier version of this notebook dictated
`V_shock = V_d · r/(r−1)`, and that formula is wrong here.** It assumes the upstream
plasma is at rest, whereas it is flowing at ~411 km/s. The shipped recipe derives
the shock speed from mass-flux conservation, `V_sh = (n_d·V_d − n_u·V_u)/(n_d − n_u)`,
and gets 579 km/s where the dictated form gives 838. The Mach number was wrong for a
second reason: it divided the spacecraft-frame speed by V_A, when M_A needs the shock
speed *in the upstream plasma frame*.

**Expected, measured from the archive at the 04:01 UT crossing:**

| Quantity | Measured | Note |
|:---|:---|:---|
| Density compression r_n | 2.59 | |
| Magnetic compression r_B | 2.53 | Independent — agreement is the check |
| V_A upstream | 52.3 km/s | Low-β solar wind |
| c_s upstream | 36.5 km/s | T₁ = 96.8 kK |
| V_shock, spacecraft frame | 579 km/s | Mass-flux conservation |
| V_shock, upstream frame | 167 km/s | 579 − 411; this is what M_A uses |
| M_A | 3.20 | Moderately strong, well short of the limit |

**How you know the Mach number is right.** MHD ties compression to Mach number: at
M_A = 3.20 it predicts r = 3.10, against a measured 2.59 — the right order, the
residual being the quasi-perpendicular approximation and the sonic contribution. The
old M_A = 16 predicts r = 3.94, essentially the strong-shock limit of 4, which the
measured 2.59 flatly contradicts. **A wrong Mach number is not detectable on its own;
it is detectable against the compression.** That is why the prompt asks for the check
rather than for the answer.

In [ ]:
%%helioai
Now the shock physics. Do a Rankine-Hugoniot analysis of the crossing
you just characterised.

I want the compression ratio, the upstream Alfven and sound speeds, the
shock speed and the Alfvenic Mach number. One thing to be careful about:
the upstream plasma is flowing at ~400 km/s, it is not at rest, so derive
the shock speed in the frame the measurement was actually made in and be
explicit about which frame each number lives in.

Then check your own work before you interpret anything. The density and
the magnetic compression are two independent measurements of the same
jump, so they should agree. And the compression ratio and the Mach number
are not independent, since MHD ties them together. If your numbers do not
satisfy that relation, say so and tell me which one you trust.

Finally: how strong is this shock really, how close to the strong-shock
limit, and what does that imply for diffusive shock acceleration of
energetic particles?

---
## Act IV — Multi-spacecraft Shock Timing

In March 2015 both ACE and Wind sat near L1, tens of R_E apart. Two spacecraft, one delay: this act is about what that pair can honestly establish, and what it cannot.

**Measured positions at shock arrival** — from `amda/ace_xyz_gse` and `amda/wnd_xyz_gse`, in R_E. The prompt asks the agent to fetch these, not to read them here:

| | X | Y | Z |
|:---|---:|---:|---:|
| ACE | 221.5 | −10.8 | −23.4 |
| Wind | 253.1 | 54.5 | 12.6 |
| ΔR (ACE − Wind) | −31.6 | −65.2 | −36.0 |

Wind is **31.6 R_E further sunward**, so it should see the shock first. Measured lag: **+270 s**, ACE behind Wind — the right sign, and the first thing to check.

**Time the crossing on the 3-second product, not the 1-minute one.** `WI_H0_MFI/B3GSM` puts the Wind ramp at 03:59:55–04:00:07; `WI_H0_MFI/BGSM` at 60 s can only place it inside a minute, and a minute is the same size as a large part of the lag being measured. Act I already asks for a few-second cadence for exactly this reason.

**Two spacecraft cannot give you the orientation of the front.** One crossing pair yields one number, the delay, and a plane orientation has two free angles: the system is underdetermined, and no amount of care closes it. Four spacecraft do — that is what Cluster and MMS are for. So the normal has to come from somewhere else, and the field itself supplies it: coplanarity across the ramp, which is the `theta_bn` recipe.

**What this pair actually says**, all measured by `verify_reference_values.py`:

| Quantity | Measured | |
|:---|:---|:---|
| Lag (ACE − Wind) | +270 s | Wind at 03:59:55, ACE at 04:04:25 |
| Separation | 515 700 km | 80.9 R_E |
| Coplanarity normal | (0.81, 0.48, −0.33) | θ_Bn = 54.3°, upstream-pointing |
| Separation along the normal | 288 100 km | 56 % of it |
| Separation across the front | 427 600 km | the part timing says nothing about |
| Shock speed along the normal | 1069 km/s | from the timing |
| Shock speed from the jumps | 579 km/s | Act III, mass-flux conservation |

**The two disagree by 85 %, and that is the result.** A front travelling at 579 km/s perpendicular to the separation would take 891 s to cross it, against a measured 270 s; dividing ΔX by the lag gives 747 km/s, which is not a speed at all. The reason all three numbers conflict is the 427 600 km of transverse separation: every one of them assumes a single flat front across that distance, and interplanetary shock fronts ripple on far smaller scales. The exact figures move with the averaging windows the normal is derived from — which is itself the answer.

> **The trap that produces a confident wrong number.** Deriving the normal from the timing, `n = ΔR / (V_shock · Δt)`, is the separation vector rescaled. Every quantity computed from it is then an identity: the along-normal separation comes back as |ΔR| and the transverse separation as **exactly 0 km, for any input whatsoever**. A run of this notebook published that zero as a geometrical result, next to a shock-front orientation that was really just the ACE→Wind direction. The `shock_timing_2sc` recipe takes the normal as a required argument for this reason, and says out loud when the transverse separation exceeds the along-normal one.

> Note the ephemeris trap the prompt warns about. `WI_OR_DEF/GSE_POS` covers 1994–1997 and `WI_K0_3DP/sc_position` starts in 2019; both return nothing for 2015. An earlier run took the empty result as licence to estimate Wind's position, and produced a negative apparent speed from it. `amda/wnd_xyz_gse` works.


In [ ]:
%%helioai
Last piece: the two-spacecraft timing.

Take the window 2015-03-17T03:30 to 05:00 UT and overlay the field
magnitude from Wind and from ACE. Work from the field rather than the
density: both magnetometers have essentially full coverage here, and
the shock is a clean step in B. Use the few-second Wind product, not
the one-minute one — a minute is the same size as the lag I am asking
you to measure.

Measure the arrival time at each spacecraft and the lag between them.
Then fetch where the two actually were, in GSE, at that moment. Fetch
it, do not assume it, and if an ephemeris product has no data for 2015,
try another rather than estimating a position.

Now the geometry, and this is where I want you to be careful. Two
spacecraft cannot give you the orientation of the front: one delay is
one number, an orientation has two angles. So do not work a normal out
of the separation and the lag — that is circular, and it will hand you
a transverse separation of exactly zero no matter what the data says.
Get the normal from the field's own structure across the crossing
instead.

With a normal in hand, the timing gives you the shock speed along it,
which Act III already gave you a different way. Compare the two. If they
disagree, tell me by how much, which of the assumptions behind them you
would drop first, and what you would need to settle it.


---
## Act V — Instant Plasma Physics Reference

For quick sanity checks, call PlasmaPy tools **directly** — zero latency, no LLM call, no API key needed.  
Useful before an agent session to define expected value ranges, or after to verify the agent's numbers.

In [ ]:
from helioai.tools.plasmapy_tools import (
    alfven_speed,
    debye_length,
    gyrofrequency,
    inertial_length,
    plasma_beta,
)

# St Patrick's Day 2015: three regimes, each a measured Wind average over the
# interval named in the label (MFI for B, SWE H1 moments for n and T).
regions = {
    "Quiet upstream   (Mar 16 22-23 UT)": {"B_nT": 8.3, "n_cm3": 17.7, "T_eV": 9.9},
    "Shocked sheath   (Mar 17 04-05 UT)": {"B_nT": 24.4, "n_cm3": 43.8, "T_eV": 21.0},
    "Southward-Bz driver (Mar 17 12-14 UT)": {"B_nT": 30.7, "n_cm3": 24.8, "T_eV": 32.2},
}

hdr = f"{'Region':<38}  {'beta':>5}  {'VA km/s':>9}  {'fci Hz':>8}  {'lD m':>7}  {'di km':>7}"
print(hdr)
print("-" * len(hdr))

results = {}
for label, p in regions.items():
    b = await plasma_beta(p["B_nT"], p["n_cm3"], p["T_eV"])
    va = await alfven_speed(p["B_nT"], p["n_cm3"])
    fci = await gyrofrequency(p["B_nT"], "proton")
    ld = await debye_length(p["n_cm3"], p["T_eV"])
    di = await inertial_length(p["n_cm3"], "proton")
    results[label] = {"b": b, "va": va}
    print(
        f"{label:<38}  {b['beta']:>5.2f}  "
        f"{va['alfven_speed_km_s']:>9.1f}  "
        f"{fci['frequency_Hz']:>8.4f}  "
        f"{ld['debye_length_m']:>7.2f}  "
        f"{di['inertial_length_km']:>7.1f}"
    )

print()
up_key = "Quiet upstream   (Mar 16 22-23 UT)"
sheath_key = "Shocked sheath   (Mar 17 04-05 UT)"
ups, down = regions[up_key], regions[sheath_key]
va1 = results[up_key]["va"]["alfven_speed_km_s"]
print(f"Density compression n_sheath/n_up : {down['n_cm3'] / ups['n_cm3']:.1f}x")
print(f"B-field compression B_sheath/B_up : {down['B_nT'] / ups['B_nT']:.1f}x")
print(f"Upstream V_A                      : {va1:.1f} km/s")
print("Strong-shock limit (g=5/3)        : r_max = 4.0")
print()
print("These are hour-long averages, so they differ from the sharp")
print("upstream/downstream jump in Act III, so expect the same order, not")
print("the same number.")

---
## Act VI — Reproducible Export

Every `run_python` call from the agent is **automatically saved** as a `code_N.py` script alongside the figures in the workspace.  
Ask the agent to bundle the full session into a standalone script — runnable with only `speasy`, `numpy`, `matplotlib`, and `plasmapy`. No HelioAI, no API key.

This is the artefact you attach to a paper or share with a colleague.

In [ ]:
%%helioai
Bundle this whole session into one standalone Python script I can hand to
a colleague. Nothing but speasy, numpy, matplotlib and plasmapy: no
HelioAI import, no API key, so it still runs in five years.

It should reproduce what we did: download the data, make the four-panel
figure with the shock marked, print the Rankine-Hugoniot table and the
two-spacecraft timing result.

Two things I want done properly. Handle the fill values the way we did
here, from each variable's declared FILLVAL. A hard-coded cutoff gets
Wind or ACE wrong, because they use different sentinels. And put every
choice that defines the event in one CONFIG block at the top: dates,
parameter ids, output path. I want to point this at a different storm by
editing that block and nothing else.

Header comment with the event, the instrument references (Lepping 1995
for Wind/MFI, Ogilvie 1995 for Wind/SWE, Smith 1998 for ACE/MAG), the Dst
source, and the speasy version it was written against.

Save it as stpatrick_2015_standalone.py in the workspace.

---
## What you just did — and what took hours before HelioAI

| Task | Old workflow | HelioAI |
|:---|:---|:---|
| Find correct speasy IDs | 30 min CDAWeb browsing | ~30 s |
| Write download + plot code | 1–2 h Python/IDL | ~60 s |
| Compute Rankine-Hugoniot | 30 min manual formulas | ~60 s |
| Multi-spacecraft timing | 1 h coordinate geometry | ~90 s |
| Export reproducible script | 30 min refactoring | ~30 s |
| **Total** | **3–5 h** | **< 10 min** |

Not one prompt above contained a formula. The agent was given the event, the
constraints that matter and the traps it could not know about, and had to reach for
the physics itself — `list_recipes` and `load_recipe` expose a `rankine_hugoniot`
recipe, among nine others.

### The part worth keeping

Four traps here are not incidental. Each one silently produces a plausible wrong
answer, which is the only kind that matters:

- **Fill values are not NaN, and the sentinel is not universal.** Wind/SWE fills with
  99999.9, ACE with −1e31. A `|x| > 1e30` filter passes Wind's straight into your
  averages as a solar-wind speed. Read `FILLVAL` from the metadata. A blanket
  "reject ≥ 99999" rule is also wrong — OMNI carries a real proton temperature of
  99093 K during the 2003 Halloween window.
- **`K0` means browse quality.** No fill values *and* spikes to 166 cm⁻³, which put a
  false shock four hours early and a compression ratio of 1.1.
- **A density jump is not a shock.** Require n, |B| and V to step up together, or the
  12:51 UT compression structure will be misread as the shock.
- **A method that cannot resolve the question will answer it anyway.** Two spacecraft
  cannot determine a shock normal, so a run derived one from the separation and the lag —
  which is circular, and returns a transverse separation of exactly 0 km whatever the data.
  It was published as a geometrical result. Act IV asks for the cross-check that exposes it.
- **A number that cannot be cross-checked cannot be trusted.** The Mach number here
  was wrong by a factor of five in an earlier version, and nothing about M_A = 16 looks
  wrong on its own. It is only visible against the compression ratio, which MHD ties
  to it. Every quantity in this notebook that could be derived two ways, was.

### On the event that is not here

This notebook was written for the 2003 Halloween storm and moved after the coverage
was measured rather than assumed: ACE/SWEPAM is 100 % fill for that window, Wind/SWE
69–74 %, OMNI 1-min 57 %, and SOHO/CELIAS clips its speed at 1019 km/s. The literature
values that were in the old reference table — 70–100 cm⁻³, 1850–2000 km/s — cannot be
reproduced from any standard speasy product, and were attributed to an instrument with
no data for those dates. Extreme events break instruments; that is worth knowing, and
it is why a first demonstration should not depend on one.

---

**References** — instrument and index sources only. Event-specific citations were
removed rather than reconstructed from memory; add your own.

- Lepping et al. (1995), *Space Sci. Rev.*, 71, 207 — Wind MFI magnetometer
- Ogilvie et al. (1995), *Space Sci. Rev.*, 71, 55 — Wind SWE solar wind experiment
- Smith et al. (1998), *Space Sci. Rev.*, 86, 613 — ACE magnetic field instrument
- [WDC for Geomagnetism, Kyoto](https://wdc.kugi.kyoto-u.ac.jp/dstdir/) — Dst index

**Links**
- [HelioAI on GitHub](https://github.com/erdoganfurkan/HelioAI)
- [speasy documentation](https://speasy.readthedocs.io)
- [PlasmaPy documentation](https://docs.plasmapy.org)
- [Wind project](https://wind.nasa.gov/)
- [ACE Science Center](https://www.srl.caltech.edu/ACE/ASC/)